# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


In [1]:
import Pkg
Pkg.add("Distributions")
Pkg.add("JuMP")
Pkg.add("MiniZinc")
Pkg.add("MathOptInterface")
Pkg.add("DataStructures")
Pkg.add("Distributions")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
   Resolving package versions...
     Project No packages ad

## Generowanie plików z problemami

In [28]:
using Distributions

function make_dzn(n::Int, id::Int, R::Int = 100, α::Float64 = 0.5)
    weights = rand(DiscreteUniform(R / 10, R), n)

    profits = [rand(DiscreteUniform(R / 10, weight)) + R / 10 for weight in weights]
    profits = Array{Int}(profits)

    # capacity is typically some α times sum of weights
    capacity = Int64(round(α * sum(weights)));
    content = """
    ITEM = _(1..$n);
    capacity = $capacity;
    profits = $profits;
    weights = $weights;
    """
    file = open("knapsack_$id.dzn", "w+")
    write(file, content)
    close(file)
end

make_dzn (generic function with 3 methods)

In [29]:
n_items = [10, 10, 10, 50, 50, 100]

for (id, i) in enumerate(n_items)
    make_dzn(i, id)
end

In [ ]:
for i in 1:length(n_items)
    result = read(`minizinc --solver gecode my_knapsack.mzn knapsack_$i.dzn` , String)
    println(result)
end

taken = [true, false, false, false, true, true, true, false, false, true];
profit = 243;
weight = 243;
----------

taken = [false, false, true, true, true, true, true, true, true, true];
profit = 281;
weight = 281;
----------

taken = [true, true, true, false, true, false, false, true, true, false];
profit = 291;
weight = 291;
----------



## Symulowane Wyżarzanie

In [20]:
using DataStructures
using Distributions

mutable struct SAState{P, TF}
    best_seen::P
    best_seen_obj::TF
    current::P
    current_obj::TF
    considered::P
    temp::Float64
    iter::Int
end

function SAState(p, x0; initial_temp::Float64=100.0)
    obj = objective(p, x0)
    return SAState{typeof(x0), typeof(obj)}(
        copy(x0), obj, copy(x0), obj, copy(x0), initial_temp, 1
    )
end

function solve_sa(p, s::SAState; iteration_limit::Int=1000, cooling_rate::Float64=0.95)
    while s.iter < iteration_limit
        move = random_move(p, s.current) 
        
        copyto!(s.considered, s.current)
        apply!(s.considered, move)
        trial_obj = objective(p, s.considered)
    
        ΔE = trial_obj - s.current_obj
        
        if ΔE < 0 || rand() < exp(-ΔE / s.temp)
            copyto!(s.current, s.considered)
            s.current_obj = trial_obj
            
            if s.current_obj < s.best_seen_obj
                copyto!(s.best_seen, s.current)
                s.best_seen_obj = s.current_obj
            end
        end
        
        s.temp *= cooling_rate
        s.iter += 1
    end
    return s.best_seen
end


struct KnapsackProblem
    capacity::Int
    weights::Vector{Int}
    profits::Vector{Int}
end

function random_move(p::KnapsackProblem, x::Vector{Bool})
    i = rand(1:length(x))
    
    current_weight = sum(p.weights .* x)
    
    if x[i]
        return (:remove, i)
    else
        if current_weight + p.weights[i] <= p.capacity
            return (:add, i)
        else
            in_bag_indices = findall(x)
            if isempty(in_bag_indices)
                return (:add, i)
            end
            return (:remove, rand(in_bag_indices))
        end
    end
end

function objective(p::KnapsackProblem, x)
    return -sum(p.profits .* x)
end


function apply!(x, move::Tuple{Symbol,Int})
    if move[1] === :add
        x[move[2]] = true
    else
        x[move[2]] = false
    end
    return x
end



apply! (generic function with 1 method)

In [21]:
function generate_problem()
    n_items = 100
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    kp = KnapsackProblem(3000, profits, weights)
end

kp1 = generate_problem()


KnapsackProblem(3000, [649, 239, 974, 138, 418, 51, 184, 573, 234, 862  …  510, 636, 584, 480, 851, 143, 852, 635, 834, 202], [51, 12, 14, 23, 57, 54, 50, 86, 80, 99  …  16, 51, 80, 51, 32, 51, 96, 41, 64, 78])

In [22]:
function test_sa(kp)
    x0 = fill(false, length(kp.weights))
    
    st = SAState(kp, x0; initial_temp=1000.0) 
    
    sol = solve_sa(kp, st; iteration_limit=1000000, cooling_rate=0.99999)
    
    println("Items selected (indices): ", findall(sol))
    println("Best objective (Profit): ", -st.best_seen_obj)
    println("Total Weight: ", sum(kp.weights[sol]))
    println("Last iteration: ", st.iter)
end

test_sa (generic function with 1 method)

In [23]:
test_sa(kp1)

Items selected (indices): [6, 9, 12, 18, 19, 20, 26, 27, 34, 47, 51, 52, 56, 59, 60, 66, 67, 71, 81, 89, 96, 100]
Best objective (Profit): 1293
Total Weight: 2991
Last iteration: 1000000
